# Demo 06 : Tracing with LangSmith

Run the cells in order and watch. There is nothing to write.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


## a · trainer

We enable tracing inside this trainer demo only. The endpoint comes from the EU-region `.env` setting, and the surrounding environment keeps its default tracing setting.

In [2]:
import json
import sqlite3

from langchain.tools import tool

DB = data("demo-world/service.db")


@tool
def bus_info(fleet_no: str) -> str:
    """Look up a Veenhaven bus model and depot by its fleet number."""
    with sqlite3.connect(DB) as conn:
        row = conn.execute(
            "SELECT model, depot_id FROM buses WHERE fleet_no = ?", (fleet_no,)
        ).fetchone()
    return json.dumps(dict(zip(["model", "depot_id"], row))) if row else "Unknown bus"


@tool
def parts_for_model(model: str) -> str:
    """List spare parts, stock and storage depots for a bus model such as CE-12."""
    with sqlite3.connect(DB) as conn:
        rows = conn.execute(
            "SELECT part_no, name, stock, depot FROM parts WHERE model = ?", (model,)
        ).fetchall()
    return json.dumps([dict(zip(["part_no", "name", "stock", "depot"], row)) for row in rows])


tools = [bus_info, parts_for_model]
question = {"messages": [("human", "Which door actuator fits bus CE12-2021-0117, and is it in stock? Look up the bus first.")]}

In [3]:
import os
import time
from uuid import uuid4
from langchain.agents import create_agent
from langsmith import Client, trace, tracing_context

PROJECT = "ARI520-module04-building-agents"
print("project:", PROJECT)
print("endpoint:", os.getenv("LANGSMITH_ENDPOINT", "not configured"))
print("default tracing:", os.getenv("LANGSMITH_TRACING", "not configured"))
agent = create_agent(llm, tools=tools,
                     system_prompt="Look up the bus before its parts. Use the database for stock.")
out = None
try:
    if not os.getenv("LANGSMITH_API_KEY") or not os.getenv("LANGSMITH_ENDPOINT"):
        raise RuntimeError("LangSmith key or EU endpoint is missing")
    client = Client(api_url=os.environ["LANGSMITH_ENDPOINT"],
                    api_key=os.environ["LANGSMITH_API_KEY"], timeout_ms=10000)
    list(client.list_projects(limit=1))
    with tracing_context(enabled=True, client=client, project_name=PROJECT):
        with trace("door-actuator-stock", run_type="chain", inputs=question,
                   run_id=uuid4()) as root_run:
            out = agent.invoke(question, {"recursion_limit": 16})
            root_run.end(outputs={"answer": out["messages"][-1].content})
    client.flush()
    project = next(client.list_projects(name=PROJECT, limit=1))
    for attempt in range(3):
        try:
            recorded = await client.runs.retrieve(str(root_run.id), project_id=str(project.id))
            break
        except Exception:
            if attempt == 2:
                raise
            time.sleep(0.5)
    print("trace recorded:", recorded.id)
except Exception as exc:
    print("tracing failed:", type(exc).__name__)
if out is None:
    with tracing_context(enabled=False):
        out = agent.invoke(question, {"recursion_limit": 16})
print("answer:", out["messages"][-1].content)
print("default tracing after:", os.getenv("LANGSMITH_TRACING", "not configured"))

project: ARI520-module04-building-agents
endpoint: https://eu.api.smith.langchain.com
default tracing: false


trace recorded: 9b7409c0-99da-4308-9580-bf548e1b91cf
answer: Bus CE12-2021-0117 is a CE-12. Its door actuator is part **CE-DA-03**; it is **out of stock** (0 available at Rietpoort).
default tracing after: false


We inspect the named project in LangSmith for the model calls, tool arguments, results and timing. If recording fails, the answer still runs and the cell prints the failure type.

## Things to try

Optional : change the bus or part identifier and run the example again.